<img src="../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../images/Konstanz_Logo.svg" width="200" /> <img src="../images/KIT_Logo.png" width="200" />

# Introduction to Dask

Every library so far in this workshop has made the same two assumptions: that your
data fits into the memory of one machine, and that one machine is all you have.
**Dask** drops both.

It manages that without asking you to learn a new way of thinking. A Dask Array
behaves like a NumPy array, a Dask DataFrame behaves like a pandas DataFrame — the
same methods, the same attributes, the same results. Underneath, the data is cut
into many small NumPy arrays or pandas DataFrames, and the operations you ask for are
carried out on those pieces in parallel: across the cores of one node, or across
several nodes of the cluster.

![image](../images/Dask_Scale.svg)

This notebook introduces the two collections you will use most, on data small enough
to check by eye. The second notebook,
[`02_Dask_On_The_Cluster.ipynb`](02_Dask_On_The_Cluster.ipynb), then takes the same ideas to
several compute nodes of the bwUniCluster.

---

## Contents

1. [Why Dask?](#why)
2. [The Client and the Dashboard](#client)
3. [Dask Array](#array)
4. [Dask DataFrame](#dataframe)
5. [When Dask is the wrong tool](#wrong-tool)

<a id="why"></a>
## 1. Why Dask?

Two separate limits appear as datasets grow, and Dask is built to address both.

**The first limit is memory.** `pd.read_parquet` loads an entire table into RAM. A
file bigger than the memory your job was allocated cannot be opened that way at all —
the process is killed. Dask instead splits the table into **partitions** and keeps
only a few of them in memory at any moment, streaming the rest from disk as it goes.
Data far too large for pandas becomes workable, at the price of reading it more than
once.

**The second limit is that one process uses one core.** A compute node on the
bwUniCluster offers dozens of cores, and a plain pandas or NumPy operation uses one
of them. Dask expresses your computation as a graph of independent **tasks** and
hands those tasks to a pool of **workers** that run them at the same time.

Both follow from the same design decision: Dask does not reimplement NumPy and
pandas, it *coordinates* them. Every chunk of a Dask Array really is a NumPy array,
and every partition of a Dask DataFrame really is a pandas DataFrame. That is why the
API feels familiar, and it is also why everything you learned in chapters 03 and 04
still applies here.

### 1.1 The vocabulary

Five words come up constantly, and the rest of the notebook is much easier once they
are clear:

- a **collection** is the object you work with — a Dask Array, a Dask DataFrame
- a **chunk** (arrays) or **partition** (dataframes) is one piece of that collection,
  and is itself an ordinary NumPy array or pandas DataFrame
- a **task** is a single unit of work, such as *"read partition 3"* or
  *"take the mean of this chunk"*
- the **task graph** is all the tasks plus the dependencies between them: the recipe
  for your result
- the **scheduler** hands tasks out, and the **workers** carry them out

The consequence is the one habit you have to acquire: Dask operations are **lazy**.
Asking for a mean does not give you a number, it adds tasks to a graph and gives you
back a collection. Nothing runs until you say so with `.compute()`. Section 3 shows
exactly what that looks like.

<a id="client"></a>
## 2. The Client and the Dashboard

Dask runs without any setup — it falls back to a thread pool on the local machine.
But creating a `Client` from `dask.distributed` is worth the one line, because it
gives you explicit control over how many workers exist and how much memory each one
may use, and it starts the **Dask Dashboard**: a live view of which tasks are
running, how busy each worker is, and how much memory is in use.

Displaying the client object prints the dashboard's URL.

In [ ]:
from dask.distributed import Client

client = Client(processes=False,        # run the workers as threads in this process
                n_workers=1,
                threads_per_worker=4,  # four tasks at a time
                memory_limit='4GB')    # ... within 4 GB
client

`processes=False` asks for **threads** rather than separate processes. For the work in
this notebook that is the better choice: NumPy and pandas spend their time inside
compiled C code, which releases Python's Global Interpreter Lock, so several threads
really do run at once — and threads in one process share their data, so no copying is
needed to move a chunk between them.

Pure Python code is the opposite case: the GIL prevents threads from running in
parallel, and processes are needed instead. The cluster notebook returns to this
trade-off when it chooses how many workers and threads to place on each node.

### 2.1 Reaching the dashboard from your own computer

The URL above points at a port on the compute node your JupyterLab job is running on,
which your laptop cannot reach directly. Forward the port over ssh first — in a
terminal **on your own computer**, not in Jupyter:

```bash
ssh -N -L <Port>:<Jupyter-Compute-Node>:<Port> <university abbreviation>_<User-ID>@bwunicluster.scc.kit.edu
```

Both the port and the node name are in the URL printed above. For example:

```bash
ssh -N -L 8787:10.0.1.112:8787 es_pkoester@bwunicluster.scc.kit.edu
```

While that command runs, the dashboard is available locally at
[http://localhost:8787/status](http://localhost:8787/status). Keep it open on a second
screen for the rest of this notebook — the graphs make the difference between lazy and
computed much easier to see than any printout.

<a id="array"></a>
## 3. Dask Array

A Dask Array is a **grid of NumPy arrays**. You describe one large logical array; Dask
stores it as many small ones and applies every operation to each of them separately.

Which parts of the NumPy API are available is listed in the documentation:
[https://docs.dask.org/en/latest/array-api.html](https://docs.dask.org/en/latest/array-api.html)

### 3.1 Creating an array

The `chunks` argument is the only thing that is new. It says how the array is to be
cut up — here into blocks of 2000 x 2000 elements.

In [ ]:
import dask.array as da

x = da.random.random((10000, 10000), chunks=(2000, 2000))
x

Look closely at what was printed: shape, data type, chunk layout — and **not a single
value**. A 10000 x 10000 array of random numbers was not created. What exists is the
instruction to create it, in 25 independent pieces.

In [ ]:
print("shape:            ", x.shape)
print("one chunk:        ", x.chunksize)
print("grid of chunks:   ", x.numblocks)      # 5 x 5 = 25 chunks
print("total size in GB: ", round(x.nbytes / 1e9, 2))

### 3.2 Lazy evaluation

Operations behave the same way. Taking a mean along an axis is a NumPy one-liner, and
in Dask it returns another Dask Array rather than the answer.

In [ ]:
y = x.mean(axis=0)
y

Still no numbers — just a smaller recipe built on top of the previous one. `.compute()`
is what finally runs it. Everything happens at that moment: the chunks are created,
each one is reduced, and the partial results are combined.

Watch the dashboard while this cell runs.

In [ ]:
result = y.compute()

print(type(result))         # a plain NumPy array comes back
print(result[:5])

That is the rule in full: **Dask collections in, Dask collections out, until
`.compute()` turns the last one into ordinary NumPy or pandas.**

The habit this asks for is to build up the whole calculation first and call
`.compute()` once at the end. Calling it early, in the middle of a chain, throws away
Dask's chance to optimise across the steps — and calling it on something that does not
fit in memory defeats the purpose entirely, because the result has to come back to
this one process.

### 3.3 Seeing the task graph

"A recipe" is easier to believe once you have looked at one. Every Dask collection can
draw its own task graph with `.visualize()`, and on a deliberately tiny array the
result is small enough to read end to end.

In [ ]:
small = da.random.random((20, 20), chunks=(10, 10))

small.visualize()

Four chunks, so four independent tasks, and nothing connecting them — each circle is a
function to call, each square a piece of data it produces. Because none of them depends
on another, all four can run at the same time on four different workers.

Now add a reduction on top.

In [ ]:
small.sum().visualize()

The graph gained a second layer: each chunk is summed on its own, and those four partial
sums are then combined into the single final value. This is why a `sum` over an array
larger than memory works — the partial results are tiny, and each chunk can be released
as soon as it has been reduced.

That two-stage shape is worth remembering, because it is the same one a `groupby` on a
Dask DataFrame uses in [section 4](#dataframe).

> `.visualize()` needs the `graphviz` Python package *and* the `dot` program it drives.
> Both are available on the bwUniCluster, and `graphviz` is part of this workshop's
> environment. The call also leaves a `mydask.png` file next to the notebook.

Graphs like these get unreadable fast — the 25-chunk array from section 3.1 already
produces a wall of boxes, and the arrays in section 3.5 have hundreds of chunks. Use
`.visualize()` on a small version of a problem to understand its shape, then scale up.

### 3.4 Choosing the chunk size

Chunks are the one tuning decision a Dask Array asks of you, and both extremes are
bad. Very small chunks mean a huge number of tiny tasks, and the scheduling overhead
per task starts to dominate the actual arithmetic. Very large chunks mean few tasks —
too few to keep the workers busy — and each one needs a lot of memory.

Something in the region of 100 MB per chunk is a reasonable default, and Dask accepts
that as a size directly instead of a shape.

In [ ]:
x2 = da.random.random((10000, 10000), chunks="100 MiB")

print("chunk shape:", x2.chunksize)
print("chunks:     ", x2.numblocks)

### 3.5 An array larger than memory

This is the part pandas and NumPy cannot do. The array below is deliberately bigger
than the 4 GB the worker is allowed to use — `np.random.random((30000, 30000))` would
fail with a `MemoryError`.

In [ ]:
big = da.random.random((30000, 30000), chunks="100 MiB")

print("size in GB:", round(big.nbytes / 1e9, 2))
print("chunks:    ", big.numblocks)

In [ ]:
%%time
# %%time measures how long the cell takes and must be its first line

big.mean().compute()

It worked, and the memory graph on the dashboard stayed flat rather than climbing to
7 GB. The reason is that a mean never needs the whole array at once: each chunk is
created, reduced to a running total, and thrown away again before the next one is
started. Dask only ever holds as many chunks as there are threads.

This is also why `big.compute()` on its own would be a bad idea — it would ask for all
7 GB to be collected back into this process. Reductions are safe; materialising is not.

### 3.6 Only the chunks you need

Because nothing runs until `.compute()`, Dask can see what you actually asked for and
skip the rest. Slicing a corner out of that same 7 GB array touches exactly one chunk.

In [ ]:
%%time

big[:3, :3].compute()

Milliseconds, not seconds. All the other chunks -- every one but this single block -- were never created — the same
laziness that felt like an inconvenience in section 3.2 is what makes this possible.

<a id="dataframe"></a>
## 4. Dask DataFrame

What a Dask Array is to NumPy, a **Dask DataFrame** is to pandas: a sequence of
pandas DataFrames, split **by rows**, that together behave like one large table. Each
partition holds all the columns and some of the rows.

The available operations are documented at
[https://docs.dask.org/en/latest/dataframe-api.html](https://docs.dask.org/en/latest/dataframe-api.html).

We use the same New York taxi file as chapter 04, so that the comparison with pandas
is direct — one row per trip, with pickup time, distance, fare and tip.

*Data source: [https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)*

### 4.1 Building one from a pandas DataFrame

Reading it with pandas first makes the relationship between the two objects visible.

In [ ]:
import pandas as pd

pdf = pd.read_parquet('../files/green_tripdata_2023-01.parquet', engine='pyarrow')

print("rows:   ", pdf.shape[0])
print("columns:", pdf.shape[1])
pdf.head(3)

`dd.from_pandas` cuts an existing pandas DataFrame into a chosen number of partitions.
In real work you would not go through pandas at all — section 4.5 reads the file
directly — but it is the clearest way to see what a Dask DataFrame is made of.

In [ ]:
import dask.dataframe as dd

ddf = dd.from_pandas(pdf, npartitions=4)
ddf

The output shows the column names and their data types, and says `npartitions=4` — but
no rows, for the same reason the array showed no values.

In [ ]:
print("partitions:", ddf.npartitions)
print("divisions: ", ddf.divisions)     # the index value each partition starts at

`divisions` records where the partitions begin and end along the index. Dask does not
always know this — reading a file usually leaves it unknown — but when it does, it can
skip whole partitions that cannot contain the rows you asked for.

### 4.2 Every partition is a real pandas DataFrame

This is the claim from section 1, and it is worth checking rather than believing.

In [ ]:
part = ddf.partitions[0].compute()

print(type(part))
print("rows in partition 0:", len(part))
part.head(3)

An ordinary `pandas.DataFrame`, roughly a quarter of the rows. Four of these stacked
on top of each other are the whole table, and that is the entire trick: Dask keeps
track of the pieces and of what has to happen to each one.

### 4.3 The pandas API, unchanged

Because the partitions are pandas objects, the methods are pandas methods. Asking for
the mean tip returns a lazy result, exactly as with the array.

In [ ]:
ddf['tip_amount'].mean()

In [ ]:
ddf['tip_amount'].mean().compute()

And the number agrees with pandas, which is the point — Dask is not an approximation.

In [ ]:
print("dask:  ", ddf['tip_amount'].mean().compute())
print("pandas:", pdf['tip_amount'].mean())

Boolean masks from [chapter 04](../04_Pandas/01_Introduction_To_DataFrames_And_Series.ipynb#filtering)
work the same way. The filter is applied to each partition independently, which is why
it parallelises so well.

In [ ]:
long_trips = ddf[ddf['trip_distance'] > 10]

print("mean fare on trips over 10 miles:",
      round(long_trips['total_amount'].mean().compute(), 2))

A `groupby` is more interesting, because the groups do not respect partition
boundaries: the same `passenger_count` appears in all four. Dask computes a partial
result per partition and then combines them — a *split-apply-combine* in two stages,
which you never have to write out yourself.

In [ ]:
ddf.groupby('passenger_count')['tip_amount'].mean().compute().sort_index()

### 4.4 The one method that is not lazy

`head()` is deliberately eager, because its whole purpose is to let you glance at the
data. It reads the first partition only, so it stays cheap no matter how large the
table is.

In [ ]:
ddf.head(3)

### 4.5 Reading files directly

In practice you hand the file to Dask and never create a pandas DataFrame at all. This
is also the only version that works when the file does not fit in memory.

In [ ]:
ddf = dd.read_parquet('../files/green_tripdata_2023-01.parquet', engine='pyarrow')

print("partitions chosen by Dask:", ddf.npartitions)
ddf.head(3)

Dask picked the partitioning itself, from the row groups stored inside the Parquet
file, aiming at partitions of a few hundred MB. Our teaching file is only about
1.4 MB, so it lands in very few partitions — there is simply nothing to spread out.

The line that makes all of this worthwhile is the next one. `read_parquet` and
`read_csv` accept a **pattern**, and treat every file that matches as one table:

```python
ddf = dd.read_parquet('/pfs/work/<your-workspace>/taxi/green_tripdata_*.parquet')
```

Twelve monthly files, or five hundred, become a single Dask DataFrame with the same
columns and the same methods as the one above. In pandas this is a loop, a list of
frames and a `pd.concat` that has to fit in memory all at once. Here the number of
files changes nothing about the code you write after it.

### 4.6 persist(): keeping results in worker memory

Because a Dask DataFrame is only a recipe, every `.compute()` runs that recipe from
the beginning. The next cell therefore reads the file **twice**.

In [ ]:
%%time

print("mean tip:  ", round(ddf['tip_amount'].mean().compute(), 3))
print("mean total:", round(ddf['total_amount'].mean().compute(), 3))

`persist()` runs the tasks once and keeps the resulting partitions in worker memory,
handing back a collection that points at them. It returns immediately, so `wait()` is
used to hold on until the work is actually finished.

In [ ]:
from dask.distributed import wait

ddf = ddf.persist()
_ = wait(ddf)

In [ ]:
%%time

print("mean tip:  ", round(ddf['tip_amount'].mean().compute(), 3))
print("mean total:", round(ddf['total_amount'].mean().compute(), 3))

Faster, because the file is not touched again. The condition attached to `persist()` is
the obvious one: the data has to fit in the workers' combined memory. It is the right
call for a cleaned-up subset you are about to query many times — the machine learning
section of the cluster notebook uses it exactly that way — and the wrong call for a
table you will pass over once.

### 4.7 When pandas is not enough: map_partitions

Not everything you need exists in Dask's API. `map_partitions` is the escape hatch: it
runs a function you write on **each partition**, and your function receives a real
pandas DataFrame, so inside it you are simply back in chapter 04.

In [ ]:
def add_duration(df):
    """Add the trip duration in minutes. Receives one partition as a pandas DataFrame."""
    df = df.copy()
    duration = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df['duration_min'] = duration.dt.total_seconds() / 60
    return df


ddf = ddf.map_partitions(add_duration)
ddf[['lpep_pickup_datetime', 'trip_distance', 'duration_min']].head(3)

In [ ]:
print("mean trip duration in minutes:",
      round(ddf['duration_min'].mean().compute(), 2))

To build the lazy result Dask has to know the columns and dtypes your function
produces, and it works this out by running the function once on an **empty** frame. If
that fails — or gives the wrong types — describe the output yourself with the `meta`
argument.

The rule for `map_partitions` is that the function must treat each partition on its
own. Anything that needs to compare rows in different partitions cannot be written
this way.

### 4.8 Dask DataFrame compared with pandas DataFrame

Now the honest measurement. The same `groupby` on the same data, first in pandas, then
in Dask.

In [ ]:
%%time

pdf.groupby('passenger_count')['tip_amount'].mean()

In [ ]:
%%time

ddf.groupby('passenger_count')['tip_amount'].mean().compute()

**Dask is slower.** That is not a mistake in the setup, and it is the most useful
result in this notebook.

Our table is 68211 rows and about 10 MB in memory. pandas holds all of it and runs a
single optimised C loop. Dask has to build a task graph, send tasks to the scheduler,
compute a partial result per partition and merge them — and all of that costs more
than the arithmetic saved. The overhead is roughly constant; the work it saves grows
with the data. Below a certain size the overhead wins, and for this file we are well
below it.

What Dask buys you, therefore, is not speed on small data:

|                       | pandas DataFrame           | Dask DataFrame                        |
|-----------------------|----------------------------|---------------------------------------|
| where the data lives  | entirely in RAM            | partitions, a few in RAM at a time    |
| largest usable size   | your available memory      | your disk, or the whole cluster       |
| cores used            | one                        | all workers, on all nodes             |
| evaluation            | immediate                  | lazy, until `.compute()`              |
| an operation returns  | the values                 | a recipe for the values               |
| many input files      | a loop and a `concat`      | one path with a `*` in it             |
| best suited to        | up to a few GB             | tens of GB and upward                 |

Three things are worth taking away. It **removes the memory ceiling**: a table larger
than RAM can still be read, filtered and aggregated. It **uses the whole machine**,
and with the cluster notebook's setup, several machines. And it **keeps your code**:
the `groupby` above is character-for-character the pandas one plus `.compute()`, so
the work you did in chapter 04 carries straight over to data a thousand times larger.

<a id="wrong-tool"></a>
## 5. When Dask is the wrong tool

Section 4.8 is a specific case of a general rule, and knowing when *not* to reach for
Dask is as much a part of using it well as knowing the API.

**If the data fits comfortably in memory, use pandas or NumPy.** Dask's overhead is
real, and on a few million rows a single node running pandas is usually both faster and
simpler to debug. A good order to try things in: pandas first, then pandas on a node
with more memory — the bwUniCluster has nodes with a great deal of it — and only then
Dask.

**Make tasks big enough to be worth scheduling.** Each task costs roughly a
millisecond of scheduler time whatever it does, so thousands of tiny tasks spend all
their time being organised. This is the same argument as the chunk size in section 3.4,
and it is why wrapping a short Python function in Dask and calling it in a loop usually
makes things slower rather than faster.

**Watch out for operations that shuffle.** Anything that has to move rows between
partitions — `set_index`, sorting by a column, joining on a column that is not the
index — means sending data between workers, and across nodes that goes over the
network. These work, but they are expensive enough to be worth designing around: sort
once and `persist()`, rather than sorting repeatedly.

**Some things are not parallel at all.** A calculation where each step needs the
previous step's answer has nothing for a second worker to do. Dask will run it, on one
worker, more slowly than pandas would.

Finally, free the resources when you are finished. On the cluster this matters rather
more than locally: the job stays in the queue holding its nodes until it is stopped.

In [ ]:
client.close()

---

That is the whole model: collections made of NumPy arrays or pandas DataFrames, a task
graph built up lazily, and a `.compute()` that runs it. Everything else is detail.

The [second notebook](02_Dask_On_The_Cluster.ipynb) keeps all of it and changes one thing — the workers
move from threads in this process to processes spread over several compute nodes,
started through SLURM and `dask-mpi`. The `Client` is created differently; the Dask
Array and Dask DataFrame code above does not change at all.

Further reading:

- Dask Array examples: [https://examples.dask.org/array.html](https://examples.dask.org/array.html)
- Dask DataFrame best practices: [https://docs.dask.org/en/latest/dataframe-best-practices.html](https://docs.dask.org/en/latest/dataframe-best-practices.html)
- bwHPC Wiki on Python: [https://wiki.bwhpc.de/e/Python](https://wiki.bwhpc.de/e/Python)